#### NLP Exploration: Sentiment & Aspect Extraction


In [ ]:
import sys
import time
import re
from pathlib import Path
import pandas as pd
from config.config import CLEANED_REVIEWS_CSV
from collections import Counter
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [ ]:
df = pd.read_csv(CLEANED_REVIEWS_CSV)
df.shape

(254569, 8)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 254569 entries, 0 to 254568
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   city          254569 non-null  str  
 1   hotel_name    254569 non-null  str  
 2   date_raw      252982 non-null  str  
 3   title_raw     254567 non-null  str  
 4   review_raw    228783 non-null  str  
 5   title_clean   254567 non-null  str  
 6   review_clean  228783 non-null  str  
 7   date_parsed   252982 non-null  str  
dtypes: str(8)
memory usage: 15.5 MB


In [4]:
df[['city', 'hotel_name', 'date_parsed', 'title_clean', 'review_clean']].head()

,city,hotel_name,date_parsed,title_clean,review_clean
0,beijing,china_beijing_aloft_beijing_haidian,2009-10-12,Nice trendy hotel location not too bad.,I stayed in this hotel for one night. As this ...
1,beijing,china_beijing_aloft_beijing_haidian,2009-09-25,Great Budget Hotel!,Stayed two nights at Aloft on the most recent ...
2,beijing,china_beijing_aloft_beijing_haidian,2009-08-04,Excellent value - location not a big problem.,We stayed at the Aloft Beijing Haidian for 5 n...
3,beijing,china_beijing_aloft_beijing_haidian,2009-07-17,Stylish clean reasonable value poor location,I am glad to be the first person to post photo...
4,beijing,china_beijing_aloft_beijing_haidian,2009-05-30,Remote but excellent value for money,Stayed there for one night. The hotel is locat...


In [6]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\pksju\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [7]:
analyzer = SentimentIntensityAnalyzer()

In [8]:
## Trying VADER on a few sample reviews first 
sample_texts = [
    "The room was clean and the staff were incredibly friendly. Loved it!",
    "Terrible experience. Dirty room, rude staff, would not recommend.",
    "It was an okay stay, nothing special but nothing bad either.",
]

for text in sample_texts:
    scores = analyzer.polarity_scores(text)
    print(f"{scores['compound']:+.3f}  {text}")

+0.892  The room was clean and the staff were incredibly friendly. Loved it!
-0.878  Terrible experience. Dirty room, rude staff, would not recommend.
+0.557  It was an okay stay, nothing special but nothing bad either.


In [ ]:
## Try it on a handful of REAL reviews from the dataset before scaling up
sample = df['review_clean'].dropna().sample(10, random_state=1)

for text in sample:
    scores = analyzer.polarity_scores(text)
    preview = text[:120] + ('...' if len(text) > 120 else '')
    print(f"{scores['compound']:+.3f}  {preview}")

+0.909  If you are looking for good value for a stay in Montreal - I definately would recommend this hotel. The suite was roomy ...
+0.966  I spent my birthday night at this hotel and was amazed at the room size and value here. We had 5 rooms total in the "his...
+0.956  I would highly recommend this hotel to anyone, as it was one of the most pleasant experiences of my holiday. I flew in v...
+0.835  I probably called the hotel about 3 times before our arrival, and every agent I talked to was very pleasant and helpful....
+0.951  Arrived and checked in early, room wasn't ready but no problem to leave bags and head out into Oxford Street. So handy f...
+0.995  This hotel was absolutely amazing! My best friend and I spent a week in NYC for graduation, and the Woogo was the perfec...
+0.833  The hotel was well located for the Warren Street tube station. The room was clean and comfortable, but the air condition...
+0.997  We stayed for 3 nights. I was really nervous about the type of room 

In [ ]:
##  Defining a sentiment scoring function and a label bucketer
def get_sentiment(text):
    if pd.isnull(text):
        return None, None
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        label = 'positive'
    elif compound <= -0.05:
        label = 'negative'
    else:
        label = 'neutral'
    return compound, label

In [ ]:
test_batch = df['review_clean'].dropna().sample(2000, random_state=1)

start = time.time()
_ = test_batch.apply(get_sentiment)
elapsed = time.time() - start

rows_per_sec = len(test_batch) / elapsed
est_full_seconds = len(df) / rows_per_sec

print(f"2000 rows took {elapsed:.2f}s -> ~{rows_per_sec:.0f} rows/sec")
print(f"Estimated time for full {len(df)} rows: ~{est_full_seconds:.1f}s")

2000 rows took 2.30s -> ~868 rows/sec
Estimated time for full 254569 rows: ~293.2s


In [ ]:
## Running document-level sentiment on the full dataset
sentiment_results = df['review_clean'].apply(get_sentiment)
df['sentiment_score'] = sentiment_results.apply(lambda x: x[0])
df['sentiment_label'] = sentiment_results.apply(lambda x: x[1])

df['sentiment_label'].value_counts(dropna=False)

sentiment_label
positive    203151
NaN          25786
negative     24257
neutral       1375
Name: count, dtype: int64

In [ ]:
## Sanity check the distribution
df['sentiment_score'].dropna().describe()

count    228783.000000
mean          0.733386
std           0.519686
min          -0.999100
25%           0.837400
50%           0.959500
75%           0.985800
max           0.999900
Name: sentiment_score, dtype: float64

In [28]:
df.dropna(subset=['sentiment_score']).nsmallest(5, 'sentiment_score')[['hotel_name', 'sentiment_score', 'review_clean']]

,hotel_name,sentiment_score,review_clean
24385,are_dubai_arabian_courtyard_hotel_spa,-0.9991,"I don't know where to begin...For starters, th..."
242715,usa_san francisco_renoir_hotel,-0.9990,Words can't really describe how disgusting thi...
26225,are_dubai_dubai_international_hotel,-0.9986,N.B. This review is in two parts: my original ...
62719,uk_england_london_albro_house,-0.9986,"I checked into Albro House Hotel, it must have..."
51824,usa_nevada_las-vegas_plaza_hotel_casino,-0.9985,I give his hotel ZERO stars. Since it doesnt a...


In [29]:
df.dropna(subset=['sentiment_score']).nlargest(5, 'sentiment_score')[['hotel_name', 'sentiment_score', 'review_clean']]

,hotel_name,sentiment_score,review_clean
1745,china_beijing_hilton_beijing_wangfujing,0.9999,This was my wife and my first trip to both Bei...
23972,are_dubai_al_qasr_at_madinat_jumeirah,0.9999,We just arrived from a one week stay at Al Qas...
24161,are_dubai_al_qasr_at_madinat_jumeirah,0.9999,My vocabulary does not contain enough superlat...
25179,are_dubai_burj_al_arab,0.9999,The Burj is still the most amazing looking hot...
25901,are_dubai_dar_al_masyaf_at_madinat_jumeirah,0.9999,My teenage daughter and I were booked into the...


In [ ]:
## Aspect extraction - first pass (document level, keyword based)
ASPECT_KEYWORDS = {
    'room': ['room', 'bed', 'bathroom', 'shower'],
    'staff': ['staff', 'service', 'reception', 'receptionist'],
    'location': ['location', 'located', 'walk', 'distance', 'nearby'],
    'breakfast': ['breakfast', 'buffet'],
    'price': ['price', 'value', 'expensive', 'cheap', 'cost'],
    'cleanliness': ['clean', 'dirty', 'dust', 'smell'],
    'noise': ['noise', 'noisy', 'quiet', 'loud'],
}

def extract_aspects(text):
    if pd.isnull(text):
        return []
    text_lower = text.lower()
    found = [aspect for aspect, keywords in ASPECT_KEYWORDS.items()
             if any(kw in text_lower for kw in keywords)]
    return found

In [ ]:
for text in sample: ## sample contaning 10 rows with random_state = 1
    aspects = extract_aspects(text)
    preview = text[:100] + ('...' if len(text) > 100 else '')
    print(f"{aspects}  <-  {preview}")

['room', 'location', 'breakfast', 'price', 'cleanliness']  <-  If you are looking for good value for a stay in Montreal - I definately would recommend this hotel. ...
['room', 'staff', 'price', 'noise']  <-  I spent my birthday night at this hotel and was amazed at the room size and value here. We had 5 roo...
['room', 'staff']  <-  I would highly recommend this hotel to anyone, as it was one of the most pleasant experiences of my ...
['room', 'cleanliness']  <-  I probably called the hotel about 3 times before our arrival, and every agent I talked to was very p...
['room', 'staff', 'cleanliness', 'noise']  <-  Arrived and checked in early, room wasn't ready but no problem to leave bags and head out into Oxfor...
['room', 'staff']  <-  This hotel was absolutely amazing! My best friend and I spent a week in NYC for graduation, and the ...
['room', 'location', 'breakfast', 'cleanliness', 'noise']  <-  The hotel was well located for the Warren Street tube station. The room was clean and c

In [ ]:
df['aspects'] = df['review_clean'].apply(extract_aspects)

aspect_counts = Counter(a for aspects in df['aspects'] for a in aspects)
pd.Series(aspect_counts).sort_values(ascending=False)

room           206790
staff          159290
location       146401
cleanliness    108777
price          102204
breakfast       85110
noise           58326
dtype: int64

In [ ]:
## Cross-checking aspects against DOCUMENT-level sentiment - and spot the problem
for aspect in ASPECT_KEYWORDS:
    mentions = df[df['aspects'].apply(lambda a: aspect in a)]
    avg_sentiment = mentions['sentiment_score'].mean()
    print(f"{aspect:12s}  mentions={len(mentions):6d}  avg_sentiment (doc-level)={avg_sentiment:+.3f}")

room          mentions=206790  avg_sentiment (doc-level)=+0.739
staff         mentions=159290  avg_sentiment (doc-level)=+0.775
location      mentions=146401  avg_sentiment (doc-level)=+0.814
breakfast     mentions= 85110  avg_sentiment (doc-level)=+0.812
price         mentions=102204  avg_sentiment (doc-level)=+0.781
cleanliness   mentions=108777  avg_sentiment (doc-level)=+0.757
noise         mentions= 58326  avg_sentiment (doc-level)=+0.768


In [34]:
# Concrete example of the problem: pull a few reviews that mention noise and
# look at them directly - are noise complaints really scoring positive?
noise_mentions = df[df['aspects'].apply(lambda a: 'noise' in a)]
noise_mentions.nlargest(5, 'sentiment_score')[['hotel_name', 'sentiment_score', 'review_clean']]

,hotel_name,sentiment_score,review_clean
1745,china_beijing_hilton_beijing_wangfujing,0.9999,This was my wife and my first trip to both Bei...
24161,are_dubai_al_qasr_at_madinat_jumeirah,0.9999,My vocabulary does not contain enough superlat...
48852,usa_nevada_las-vegas_mgm_grand_hotel_and_casino,0.9999,We had a 12 night stay at the MGM Grand 16 - 2...
57821,usa_nevada_las-vegas_the_orleans_hotel_casino,0.9999,Absolutely loved it here! Excellent hotel. Our...
58462,usa_nevada_las-vegas_the_palms_casino_hotel,0.9999,THE PALMS IS THE BEST IN VEGAS. Let me start w...


In [ ]:
## Step 10: Fix - SENTENCE-level aspect sentiment
def split_sentences(text):
    if pd.isnull(text):
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


# Quick test
test_review = sample.iloc[0]
print(test_review)
print()
print(split_sentences(test_review))

If you are looking for good value for a stay in Montreal - I definately would recommend this hotel. The suite was roomy enough, the bathrooms were very spacious and clean. It must have just undergone renovations because everything looked so new and clean. The location is good - the street it self looks a bit rough - but it is very close to St. Catherine Street. Even though the rooms all have full kitchettes - breakfast is inlcuded in the price! Breakfast was definately NOT fancy - but fast and conveinient. Parking is also in the building for an extra fee. Although it is not luxury (the towels are really cheap) for Montreal - this is a good find and I would definately return to Les Suites Labelle.

['If you are looking for good value for a stay in Montreal - I definately would recommend this hotel.', 'The suite was roomy enough, the bathrooms were very spacious and clean.', 'It must have just undergone renovations because everything looked so new and clean.', 'The location is good - the

In [ ]:
def get_sentence_aspect_sentiments(text):
    if pd.isnull(text):
        return {}

    sentences = split_sentences(text)
    aspect_scores = {aspect: [] for aspect in ASPECT_KEYWORDS}

    for sentence in sentences:
        sentence_lower = sentence.lower()
        matched_aspects = [
            aspect for aspect, keywords in ASPECT_KEYWORDS.items()
            if any(kw in sentence_lower for kw in keywords)
        ]
        if not matched_aspects:
            continue
        compound = analyzer.polarity_scores(sentence)['compound']
        for aspect in matched_aspects:
            aspect_scores[aspect].append(compound)

    return {
        aspect: sum(scores) / len(scores)
        for aspect, scores in aspect_scores.items()
        if scores
    }


# Test on the same noise-heavy sample reviews from Step 9
for _, row in noise_mentions.head(5).iterrows():
    result = get_sentence_aspect_sentiments(row['review_clean'])
    print(f"doc-level={row['sentiment_score']:+.3f}  aspect-level={result}")

doc-level=+0.967  aspect-level={'room': 0.8883, 'location': 0.0, 'price': 0.8883, 'cleanliness': 0.8883, 'noise': 0.8883}
doc-level=+0.952  aspect-level={'staff': 0.9407, 'noise': 0.9407}
doc-level=+0.990  aspect-level={'room': 0.4486666666666667, 'staff': 0.8176, 'location': 0.2992, 'breakfast': 0.7188, 'price': 0.0, 'cleanliness': 0.0, 'noise': 0.4939}
doc-level=+0.995  aspect-level={'room': 0.62605, 'staff': 0.7269, 'location': 0.0, 'breakfast': 0.4215, 'price': 0.8313, 'cleanliness': 0.8268, 'noise': 0.0}
doc-level=+0.985  aspect-level={'room': 0.2553, 'location': 0.0, 'breakfast': 0.4215, 'price': 0.0, 'noise': 0.8625}


In [ ]:
## Timing check before running sentence-level extraction on all 254K rows
test_batch2 = df['review_clean'].dropna().sample(2000, random_state=1)

start = time.time()
_ = test_batch2.apply(get_sentence_aspect_sentiments)
elapsed = time.time() - start

rows_per_sec = len(test_batch2) / elapsed
est_full_seconds = len(df) / rows_per_sec

print(f"2000 rows took {elapsed:.2f}s -> ~{rows_per_sec:.0f} rows/sec")
print(f"Estimated time for full {len(df)} rows: ~{est_full_seconds:.1f}s (~{est_full_seconds/60:.1f} min)")

2000 rows took 0.74s -> ~2705 rows/sec
Estimated time for full 254569 rows: ~94.1s (~1.6 min)


In [ ]:
## Run sentence-level aspect sentiment on the full dataset
df['aspect_sentiments'] = df['review_clean'].apply(get_sentence_aspect_sentiments)

# Build the long-format table for SQL loading later
records = []
for review_id, aspect_dict in df['aspect_sentiments'].items():
    for aspect, score in aspect_dict.items():
        records.append({'review_id': review_id, 'aspect': aspect, 'aspect_sentiment': score})

review_aspects_df = pd.DataFrame(records)
review_aspects_df.shape

(866898, 3)

In [39]:
review_aspects_df.head(10)

,review_id,aspect,aspect_sentiment
0,0,room,0.829800
1,0,staff,0.592700
2,0,location,0.594650
3,0,breakfast,0.502300
4,1,room,0.508033
5,1,staff,0.401900
6,1,location,0.592700
7,1,breakfast,0.226300
8,1,cleanliness,0.452200
9,2,room,0.585900


In [ ]:
## Re-check aspect sentiment now at SENTENCE level - does 'noise' look right this time?
review_aspects_df.groupby('aspect')['aspect_sentiment'].agg(['count', 'mean']).sort_values('mean')

,count,mean
aspect,,
noise,58326,0.233504
room,206790,0.343885
price,102204,0.374336
breakfast,85110,0.413770
location,146401,0.424017
staff,159290,0.462467
cleanliness,108777,0.493783


In [ ]:
# Compare doc-level vs sentence-level side by side for 'noise' specifically
doc_level_noise_avg = df[df['aspects'].apply(lambda a: 'noise' in a)]['sentiment_score'].mean()
sentence_level_noise_avg = review_aspects_df[review_aspects_df['aspect'] == 'noise']['aspect_sentiment'].mean()

print(f"'noise' aspect - document-level avg sentiment: {doc_level_noise_avg:+.3f}")
print(f"'noise' aspect - sentence-level avg sentiment: {sentence_level_noise_avg:+.3f}")

'noise' aspect - document-level avg sentiment: +0.768
'noise' aspect - sentence-level avg sentiment: +0.234


In [ ]:
## Preview final structured output
df[['city', 'hotel_name', 'date_parsed', 'sentiment_score', 'sentiment_label', 'review_clean']].sample(5)

,city,hotel_name,date_parsed,sentiment_score,sentiment_label,review_clean
29930,dubai,are_dubai_jumeirah_emirates_towers_hotel,2009-09-09,-0.2411,negative,It was just awosome to stay in the hotel with ...
54218,las-vegas,usa_nevada_las-vegas_sahara_hotel_casino,2008-09-26,-0.7845,negative,"You get what you pay for - the hotel is dated,..."
237672,san-francisco,usa_san francisco_larkspur_hotel_union_square,2006-01-30,0.9866,positive,The Cartwright offers all sorts of amenities f...
25999,dubai,are_dubai_dar_al_masyaf_at_madinat_jumeirah,2007-10-29,0.9915,positive,It's taken me a while to get around to writing...
18972,chicago,usa_illinois_chicago_sofitel_chicago_water_tower,2007-01-11,0.9916,positive,My husband and I just got back from a two-nigh...


In [43]:
review_aspects_df.sample(10)

,review_id,aspect,aspect_sentiment
472005,139101,room,0.47360
695074,203702,breakfast,0.63690
542997,160900,cleanliness,0.79940
365119,109154,cleanliness,0.54990
19958,6015,staff,0.95080
119579,36577,room,0.29600
192639,59113,location,0.26680
206687,63404,breakfast,0.22630
14802,4439,staff,0.40725
323635,97036,staff,0.77830
